# Load companies into the business table

Run `populate_business_type.ipynb` first. This notebook reads `southampton_companies.csv` and populates the shared `data/processed/database.db` using Python's standard library (no extra packages required).

The requested attributes follow the existing snake_case convention:

| Column | Meaning |
| --- | --- |
| id | Companies House company number, TEXT primary key; leading zeros and letter prefixes preserved |
| business_name | Company name from company_name, with surrounding whitespace removed |
| business_type_id | Nullable foreign key to business_type.id |
| status | Company status exactly as supplied, after whitespace trimming |
| year_opened | Integer year from date_of_creation |
| year_closed | Integer year from date_of_cessation, or SQL NULL when blank |

These are company incorporation and cessation years, not verified dates of local trading activity. The CSV's Southampton selection is retained as supplied; it is not reinterpreted as a council-boundary filter.

The incorporation/dissolution summary CSVs contain yearly SIC aggregates, not company IDs. They are not used to create company rows: they cannot identify individual businesses and multi-SIC counts may count a company more than once.

**SIC policy:** this six-column table has one type per company. Use the first listed SIC code only. Exact matches to the existing five-digit dictionary are linked; missing, four-digit and unrecognised codes receive NULL. Do not pad a four-digit code to five digits: that would invent a classification. Additional codes are preserved in an import review report, not additional business rows. A future many-to-many table is needed for analysis of every SIC code. The original CSV remains the source for addresses and exact dates, which are outside this requested schema.


## 1. Settings

Run from the repository root or `data/`. The loader looks for `data/raw/official/southampton_companies.csv`, then your Downloads folder. Set `CSV_PATH` explicitly for another location. No database writes occur until section 5.


In [ ]:
import csv
import hashlib
import re
import sqlite3
from collections import Counter
from contextlib import closing
from datetime import date
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = cwd / "data" if (cwd / "data" / "populate_business_type.ipynb").is_file() else cwd
if not (DATA_DIR / "populate_business_type.ipynb").is_file():
    raise RuntimeError("Run this notebook from the repository root or data directory.")

DB_PATH = DATA_DIR / "processed" / "database.db"
local_csv = DATA_DIR / "raw" / "official" / "southampton_companies.csv"
CSV_PATH = local_csv if local_csv.is_file() else Path.home() / "Downloads" / "southampton_companies.csv"
REPORT_DIR = DATA_DIR / "processed" / "business_import"

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Set CSV_PATH to your company CSV: {CSV_PATH}")
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Run populate_business_type.ipynb first: {DB_PATH}")

with closing(sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)) as con:
    types = dict(con.execute("SELECT sic_code, id FROM business_type"))
if not types:
    raise ValueError("business_type is empty. Run populate_business_type.ipynb first.")
if any(not re.fullmatch(r"\d{5}", code) for code in types):
    raise ValueError("business_type contains SIC codes outside the expected five-digit format.")
print(f"CSV: {CSV_PATH}")
print(f"Database: {DB_PATH}")
print(f"{len(types):,} SIC codes available")


## 2. Read and validate the company records

Read all identifiers as text. Reject duplicate company numbers, malformed dates, empty business names, empty status and closure dates preceding incorporation. Blank cessation is treated as unknown; status alone does not invent a closure year. Validation errors stop the import before any business records are written.


In [ ]:
required = {"company_name", "company_number", "company_status", "date_of_creation", "date_of_cessation", "sic_codes"}
with CSV_PATH.open(newline="", encoding="utf-8-sig") as handle:
    reader = csv.DictReader(handle)
    if not required <= set(reader.fieldnames or []):
        raise ValueError(f"Missing CSV columns: {required - set(reader.fieldnames or [])}")
    source = list(reader)
if not source:
    raise ValueError("Company CSV is empty.")

def parse_date(value, label, company_number, optional=False):
    if not value and optional:
        return None
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        raise ValueError(f"{company_number}: invalid {label}: {value!r}")
    try:
        return date.fromisoformat(value)
    except ValueError as error:
        raise ValueError(f"{company_number}: invalid {label}: {value!r}") from error

rows = []
review = []
seen = set()
status_counts = Counter()
for line, raw in enumerate(source, start=2):
    if None in raw or any(raw.get(key) is None for key in required):
        raise ValueError(f"Malformed CSV row {line}: too many or too few fields.")
    company_number = raw["company_number"].strip().upper()
    if not re.fullmatch(r"[A-Z0-9]{8}", company_number):
        raise ValueError(f"Line {line}: expected an eight-character company number: {company_number!r}")
    if company_number in seen:
        raise ValueError(f"Duplicate company number: {company_number}")
    seen.add(company_number)
    business_name = raw["company_name"].strip()
    if not business_name:
        raise ValueError(f"{company_number}: blank company_name")
    status = raw["company_status"].strip()
    if not status:
        raise ValueError(f"{company_number}: blank company_status")
    opened = parse_date(raw["date_of_creation"].strip(), "date_of_creation", company_number)
    closed = parse_date(raw["date_of_cessation"].strip(), "date_of_cessation", company_number, optional=True)
    if closed is not None and closed < opened:
        raise ValueError(f"{company_number}: cessation precedes incorporation")

    sic_text = raw["sic_codes"].strip()
    codes = [code.strip() for code in sic_text.split(";")] if sic_text else []
    first_code = codes[0] if codes else ""
    business_type_id = types.get(first_code)
    reasons = []
    if not codes:
        reasons.append("missing_sic")
    elif business_type_id is None:
        reasons.append("unmatched_first_sic")
    if len(codes) > 1:
        reasons.append("multiple_sic_codes")
    if reasons:
        review.append({"company_number": company_number, "sic_codes": sic_text,
                       "selected_sic_code": first_code,
                       "business_type_id": business_type_id,
                       "reason": ";".join(reasons)})
    rows.append((company_number, business_type_id, status, opened.year,
                 None if closed is None else closed.year, business_name))
    status_counts[status] += 1

print(f"{len(rows):,} unique companies validated")
print(f"{sum(row[1] is None for row in rows):,} companies have no matched first SIC code")
print(f"{sum(row[4] is None for row in rows):,} companies have no supplied cessation date")
print("Statuses:", dict(status_counts))
rows[:5]


## 3. Save the classification review report

The report retains missing/unmatched codes and all supplied codes for multi-SIC companies. It does not discard these companies. Generated reports live under the existing Git-ignored `data/processed/` directory. The manifest records the exact input hash and selection policy; this is an import audit, not independent verification of the CSV's provenance or completeness.


In [ ]:
import json
from datetime import datetime, timezone

REPORT_DIR.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
review_path = REPORT_DIR / f"sic_review_{run_id}.csv"
with review_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        "company_number", "sic_codes", "selected_sic_code", "business_type_id", "reason"])
    writer.writeheader()
    writer.writerows(review)
manifest = {
    "input_file": str(CSV_PATH.resolve()),
    "sha256": hashlib.sha256(CSV_PATH.read_bytes()).hexdigest(),
    "prepared_at_utc": datetime.now(timezone.utc).isoformat(),
    "companies": len(rows),
    "sic_policy": "First listed SIC; exact match; unmatched or missing -> NULL",
    "year_policy": "Creation/cessation calendar years; blank cessation -> NULL",
    "review_file": str(review_path.resolve()),
    "database": str(DB_PATH.resolve()),
    "load_status": "prepared",
}
manifest_path = REPORT_DIR / f"import_{run_id}.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Classification review: {review_path}")


## 4. Schema and update rules

SQLite table `business` has the six requested attributes, including business_name. Rerunning updates the same company number and does not remove companies absent from a later file. A later blank closure date or unmatched SIC preserves an existing non-NULL value; correcting or clearing a previously known value requires an explicit reviewed update. Business name, status and opening year are refreshed from the supplied snapshot. Import snapshots in chronological order: this file has no extraction timestamp to automatically prevent an older snapshot replacing newer status.


In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS business (
    id               TEXT NOT NULL PRIMARY KEY CHECK (length(id) = 8),
    business_type_id INTEGER REFERENCES business_type(id),
    status           TEXT NOT NULL CHECK (length(trim(status)) > 0),
    year_opened      INTEGER NOT NULL
                     CHECK (typeof(year_opened) = 'integer' AND year_opened BETWEEN 1 AND 9999),
    year_closed      INTEGER
                     CHECK (year_closed IS NULL OR
                            (typeof(year_closed) = 'integer' AND year_closed BETWEEN year_opened AND 9999)),
    business_name    TEXT NOT NULL CHECK (length(trim(business_name)) > 0)
)
"""
UPSERT = """
INSERT INTO business (id, business_type_id, status, year_opened, year_closed, business_name)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(id) DO UPDATE SET
    business_type_id = COALESCE(excluded.business_type_id, business.business_type_id),
    business_name = excluded.business_name,
    status = excluded.status,
    year_opened = excluded.year_opened,
    year_closed = COALESCE(excluded.year_closed, business.year_closed)
"""


## 5. Create and populate the business table

This cell writes to the existing database. Table creation, upserts, row verification and foreign-key checks run in one transaction. An error rolls back the changes. The previous five-column schema is upgraded by adding business_name within the same transaction; matching company names are filled by the import. Existing companies absent from this CSV keep NULL names until a later import supplies them. Fresh tables require nonblank names. Other schema differences fail instead of being automatically replaced.


In [ ]:
with closing(sqlite3.connect(DB_PATH, timeout=30)) as con:
    con.execute("PRAGMA foreign_keys = ON")
    try:
        con.execute("BEGIN IMMEDIATE")
        con.execute(SCHEMA)
        columns = list(con.execute("PRAGMA table_info(business)"))
        legacy_columns = [
            ("id", "TEXT", 1), ("business_type_id", "INTEGER", 0), ("status", "TEXT", 0),
            ("year_opened", "INTEGER", 0), ("year_closed", "INTEGER", 0)]
        if [(c[1], c[2].upper(), c[5]) for c in columns] == legacy_columns:
            con.execute("""ALTER TABLE business ADD COLUMN business_name TEXT
                           CHECK (business_name IS NULL OR length(trim(business_name)) > 0)""")
            columns = list(con.execute("PRAGMA table_info(business)"))
        if [(c[1], c[2].upper(), c[5]) for c in columns] != [
            ("id", "TEXT", 1), ("business_type_id", "INTEGER", 0), ("status", "TEXT", 0),
            ("year_opened", "INTEGER", 0), ("year_closed", "INTEGER", 0),
            ("business_name", "TEXT", 0)]:
            raise ValueError("Existing business schema differs; inspect it before importing.")
        foreign_keys = list(con.execute("PRAGMA foreign_key_list(business)"))
        if not any(f[2] == "business_type" and f[3] == "business_type_id" and f[4] == "id"
                   for f in foreign_keys):
            raise ValueError("Existing business table has no expected business_type foreign key.")
        # Re-read the dictionary under the transaction; IDs must not have changed.
        if dict(con.execute("SELECT sic_code, id FROM business_type")) != types:
            raise RuntimeError("business_type changed after preparation. Rerun from section 1.")
        old = {r[0]: r for r in con.execute(
            "SELECT id, business_type_id, status, year_opened, year_closed, business_name FROM business")}
        con.execute("CREATE INDEX IF NOT EXISTS idx_business_type ON business(business_type_id)")
        con.executemany(UPSERT, rows)
        stored = {r[0]: r for r in con.execute(
            "SELECT id, business_type_id, status, year_opened, year_closed, business_name FROM business")}
        for company_number, type_id, status, opened, closed, business_name in rows:
            previous = old.get(company_number)
            expected_type = type_id if type_id is not None else (previous[1] if previous else None)
            expected_closed = closed if closed is not None else (previous[4] if previous else None)
            expected = (company_number, expected_type, status, opened, expected_closed, business_name)
            if stored.get(company_number) != expected:
                raise RuntimeError(f"Stored row differs for {company_number}")
        violations = con.execute("PRAGMA foreign_key_check(business)").fetchall()
        if violations:
            raise RuntimeError(f"Broken business foreign keys: {violations[:5]}")
        con.commit()
    except Exception:
        con.rollback()
        raise

manifest["load_status"] = "committed"
manifest["committed_at_utc"] = datetime.now(timezone.utc).isoformat()
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Loaded and verified {len(rows):,} companies into {DB_PATH}")
print(f"New companies: {sum(row[0] not in old for row in rows):,}")
print(f"Updated companies: {sum(row[0] in old for row in rows):,}")
print(f"Import manifest: {manifest_path}")


## 6. Inspect the loaded data

Counts describe the rows currently in this table. Opening/closure years retain the source's company registration meaning; neither this table nor the supplied summaries establishes trading births or deaths in Southampton.


In [ ]:
with closing(sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)) as con:
    print("Status counts:")
    for row in con.execute("SELECT status, COUNT(*) FROM business GROUP BY status ORDER BY COUNT(*) DESC"):
        print(row)
    print("Companies without a classified type:",
          con.execute("SELECT COUNT(*) FROM business WHERE business_type_id IS NULL").fetchone()[0])
    print("Sample with SIC descriptions:")
    for row in con.execute("""
        SELECT b.id, b.business_name, t.sic_code, t.description, b.status, b.year_opened, b.year_closed
        FROM business b LEFT JOIN business_type t ON t.id = b.business_type_id
        ORDER BY b.id LIMIT 10
    """):
        print(row)
